# T51 - a cyhdl port of a 8051 CPU

The T51 is a port of a legacy project from around 2002 to the cyrite ecosystem. The goal is, to allow mock-up configurations or emulations of 8051 chip derivates that no longer exist, or to run closed source binaries on custom FPGA hardware.
Hence, the provided setups don't exactly mimic a particular chip, the focus is here on co-simulation and the ability to loop in Python coroutines that create stimuli or trace execution.

Note this educational setup is work in progress. The 8051 core has been verified against the original VHDL code using a particular test binary, the arithmetic core was lock-step verified (Verification missing from this release).

This notebook is meant for learning purposes and test out small simulation setups without having to install local tools.

Note: The cyrite ecosystem is provided as Docker container only and is not considered OpenSource at this time.
This particular setup comes with the 8051 compiler `sdcc` preinstalled. Find the example hello world in `t51/example/firmware` of the project tree.

Check out the project (uncomment first):

In [1]:
# !git clone https://github.com/hackfin/t51_cyrite.git -b cyhdlport

Build the hello.bin firmware:

In [1]:
! cd t51_cyrite && make all

make -C example/firmware all
make[1]: Entering directory '/home/cyrite/t51_cyrite/example/firmware'
make[1]: Nothing to be done for 'all'.
make[1]: Leaving directory '/home/cyrite/t51_cyrite/example/firmware'


We set the necessary paths for the correct import of the subsystem:

In [2]:
import sys
sys.path.append("t51_cyrite")
sys.path.append(".")

Next, we will go through the process of setting up a simple virtual board and System On Chip.

# SoC derivation example

From the SoC architecture in `example`, we here demonstrate how to customize a working reference for own purposes. The strategy is as follows:

* See if the current architecture does the job
* If not, try a hack using available GPIO ports and custom peripherals
* If that also fails, dig one level deeper and fix the super class, accordingly

Here, you have to think a bit more like a software developer. The classical HDLs don't exactly make use of extensive object oriented programming. Hence, there are a few gotchas on how you could shoot yourself into the foot.

The actual core being mimicked in the example is a NXP 80c552. It is run in a particular setup with external memory, therefore some ports are not available and are not configureable otherwise. We will now however just start getting our hands dirty with the barebone configuration.

## Setting up a platform

The platform represents your actual board or top level design. It contains application specific pin mappings, as the `PINMAP` definition in the class below. Let us define an analog signal container first that we may want to use as external pseudo analog input signals.

In [3]:
from cyhdl import *

@container
class AnalogIn:
    _inputs = ["a0", "a1", "a2", "a3" ]
    _outputs = []
    _other = []

    def __init__(self, ctx):
        self.a0, \
        self.a1, \
        self.a2, \
        self.a3 = [ ctx.Signal(intbv()[10:]) for _ in range(4) ]

Next, we define a bare bone platform with minimum I/O. We only provide a pin map using Register type signals to mitigate human errors with association of pins. On a sidenote: We can also automatically output a C header at a later point from this pin map using an extension.

In [4]:
from t51.example.t51_factory import T51_SoC, Monitor, PipePort
from t51.example.factory import MyPort, AnalogPin, Bool, Port8051Raw
from t51.example.wishbone import WishboneBus

class BareBone(T51_SoC):

    PortSpec = T51_SoC.PortSpec
    BIDIR, IN, OUT = range(3)

    PINMAP = {
        'p1' : [
            ( 'SPI_CLK', OUT),
            ( 'SPI_MISO', IN),
            ( 'SPI_MOSI', OUT),
            ( 'SENSE', OUT),
            ( 'FLAG0', OUT),
            ( 'STAT0', IN),
            ( 'I2C_SCL', BIDIR),
            ( 'I2C_SDA', BIDIR),
        ],

        'p4' : [
            ( 'FLASH_CS', OUT),
            ( 'RTC_CS', OUT),
            ( 'RAM_CS', OUT),
            ( 'P4.3_UNUSED', OUT),
            ( 'SRO_LAT', OUT),
            ( 'SRI_LAT', OUT),
            ( 'P4.6_UNUSED', OUT),
            ( 'SRI_CLK', OUT),
        ],
    }
    
    # Override definition with our pin wiring:
    Reg_p1 = PortSpec(8, T51_SoC.regmap(PINMAP, 'p1'))
    Reg_p4 = PortSpec(8, T51_SoC.regmap(PINMAP, 'p4'))

    @cyrite_factory.block_component
    def platform_barebone(self,
        clk : ClkSignal,
        reset : ResetSignal,
        ain : AnalogIn,
        pipe : PipePort,
        seconds_tick : Bool,
        ):
        "The SoC setup for T51. Everything in here is on a hardware level."

        xram = WishboneBus(ADDR_W = 16, DATA_W = 8)

        # Ports:
        mp = [ MyPort("rp%d" % i, template = t, Signal = self.Signal) for i, t in enumerate([
            self.Reg_p0,
            self.Reg_p1,
            self.Reg_p2,
            self.Reg_p3,
            self.Reg_p4
        ]) ]

        analog_in = AnalogPin()
        
        xram_en = Bool()

        # p_o = [ Byte() for i in range(self.NUMPORTS) ]
        # p_i = [ Byte() for i in range(self.NUMPORTS) ]
        int0 = Signal(bool(False)) # No external interrupt

        # Instance the SoC.
        # Note the xram_en input. This determines which RAM port
        # is used.
        u0 = self.soc_t8052NG(
            clk = clk,
            reset = reset,
            p0 = mp[0],
            p1 = mp[1],
            p2 = mp[2],
            p3 = mp[3],
            p4 = mp[4],
            analog_in = analog_in,
            pipeio = pipe,
            xram_en = xram_en,
            xram = xram,
            ROMADDRESSWIDTH = 16,
            XRAMADDRESSWIDTH = 16
        )

        return instances()

/home/cyrite/.local/lib/python3.12/site-packages/myirl/emulation/myhdl2irl.py:909: UserWarning: /home/cyrite/t51_cyrite/t51/example/sfrperio/uart.py:uartio_model():47 Replacing logical `not` by inversion
  warnings.warn(self.get_location(node) + \
/home/cyrite/.local/lib/python3.12/site-packages/myirl/emulation/myhdl2irl.py:909: UserWarning: /home/cyrite/t51_cyrite/t51/example/sfrperio/uart.py:uartio_model():52 Replacing logical `not` by inversion
  warnings.warn(self.get_location(node) + \
/home/cyrite/.local/lib/python3.12/site-packages/myirl/emulation/myhdl2irl.py:909: UserWarning: /home/cyrite/t51_cyrite/t51/example/sfrperio/uart.py:uartio_model():62 Replacing logical `not` by inversion
  warnings.warn(self.get_location(node) + \
/home/cyrite/.local/lib/python3.12/site-packages/myirl/library/bitfields.py:51: UserWarning: Swap msb/lsb: msb was < lsb
  base.warnings.warn("Swap msb/lsb: msb was < lsb")


## The test bench

Now as we have designed the hardware portion, we would like to stick this setup into a Co-Simulation. Co-Simulation means, that a hardware model, here compiled into an executable, runs on a simulation back end, while a front end - here our Python framework - interacts with it. In the Verilog world, this type of Co-Simulation works over VPI or PLI, here, we use the coroutine concept. There are some limitations to this model:
* There is one main clock only
* No arbitrary delays can be generated in the hardware, remember, the backend drives a synthesizeable model
* The Python front end can not handle `@always_comb` or other combinatorial hierarchies, co-routines should be either a sequential process or a clock-triggered coroutine

In [5]:
import time

class MyPlatformCosim(BareBone):
    """The mock up platform with a UART coroutine
This actually just monitors a write access to the UART tx register
"""
    @cyrite_factory.testbench("ns")
    def testbench52(self,
        # signals:
    ): 
        "The Co-Simulation bench"

        # Make sure to use self.Signal:
        Signal = self.Signal
        ClkSignal = self.ClkSignal
        ResetSignal = self.ResetSignal

        clk = ClkSignal()
        reset = ResetSignal(False, False)

        sensors = AnalogIn(self)
        pipe = PipePort(self.Signal)

        tick = Signal(bool())
        
        inst_soc = self.platform_barebone(clk = clk, reset = reset,
            pipe = pipe,
            ain = sensors,
            seconds_tick = tick)

        @self.always(delay(1))
        def clkgen():
            clk.next = ~clk

        @self.sequence
        def pipeio_handling():
            "Pipe I/O handler coroutine"
            print("PIPE I/O handler listening")
            yield delay(100)
            while True:
                if pipe.en_tx:
                    pipe.busy.next = True
                    print(f"> '{chr(int(pipe.tx.evaluate()))}'")
                    yield clk.posedge
                    pipe.busy.next = False
                # Make sure to not forget any wait cycle
                yield clk.posedge

        @self.sequence
        def rtc_mockup():
            "Create a user defined seconds tick"
            while True:
                yield delay(1_000_000)
                yield clk.posedge
                tick.next = True
                yield clk.posedge
                tick.next = False

        @self.sequence
        def main():
            print("Starting simulation")
            # init:

            yield delay(20)
            reset.next = True
            print("Reset released")
            print("Running...")

            # Set pin 5 (input only)
            yield delay(20)

            print("Enter loop")
 
            yield delay(20_000)

            raise StopSimulation

        return locals()

### Running the virtual chip

First, we need to prepare an output directory for some build results:

In [6]:
! [ -e output ] || mkdir output
! [ -e runtime ] || mkdir runtime

We then choose the CXXRTLVerilog target. Direct synthesis of the t51 code is at this point not possible, because it 'violates' some particular design rules. Hence, we let yosys parse generated verilog and compile it into CXXRTL constructs.
When we run the next cell, the design is compiled into a Python module (`.so`) in `./runtime`, imported, and executed stepwise by the internal scheduler.
By running the test bench with the argument `-1`, we let the `main()` sequence take control of the simulation duration. Hence, at the end of the simulation, we should see the `StopSimulation` event.

Note also the `recompile = False` option. This prohibits the module from being recompiled every time you run the test bench. However, when you make changes to the actual hardware part, you need to set this to "True" for the time being in order to recompile the hardware model.

**Note:** In a online binder, this can take a long time to compile and you may get disconnected, depending on the load of the service. After compiling, you will get a lot of output. Make sure to re-run this cell to see the reduced output of the simulation only.

In [7]:
from yosys.simulator import CXXRTLVerilog

m = MyPlatformCosim("mine", CXXRTLVerilog, "t51_cyrite/t51/example/firmware/build/hello.bin")

tb = m.testbench52()
tb.run(-1, debug = False, wavetrace = "/tmp/t51sim.vcd", recompile = False)

 FASTCOUNT: use default False 


/home/cyrite/.local/lib/python3.12/site-packages/myirl/kernel/components.py:120: UserWarning: Fallback: Pass through other argument for arg INITDATA (<class 'bytes'>)
  base.warnings.warn(msg)
/home/cyrite/.local/lib/python3.12/site-packages/myirl/emulation/myhdl2irl.py:909: UserWarning: /home/cyrite/t51_cyrite/t51/rtl/cyhdl/t51.py:t51():t51():601 Replacing logical `not` by inversion
  warnings.warn(self.get_location(node) + \
/home/cyrite/.local/lib/python3.12/site-packages/myirl/emulation/myhdl2irl.py:761: UserWarning: Not translating unregistered decorator function`lut`
  warnings.warn("Not translating unregistered decorator function`%s`" % n)
/home/cyrite/.local/lib/python3.12/site-packages/myirl/kernel/components.py:120: UserWarning: Fallback: Pass through other argument for arg mon (<class 'NoneType'>)
  base.warnings.warn(msg)


 SECONDDPTR: use default False 
 T8032: use default False 
 TRISTATE: use default 0 
 SIMENV: use default False 
 INIT: use default True 


Attempting cold import of module `platform_barebone` in package None


Open for writing: /tmp/t51sim.vcd
PIPE I/O handler listening
Starting simulation
Reset released
Running...
Enter loop
> 'H'
> 'e'
> 'l'
> 'l'
> 'o'
> ','
> ' '
> 'w'
> 'o'
> 'r'
> 'l'
> 'd'
> '!'
' '
> '
'


STOP SIMULATION @20040


0

We note:

* The co-routine for the pipe I/O catches the UART output from the hello world program
* We get a `STOP SIMULATION` event at the end. That means, the main routine has completed.

### Exercises and ideas for things to try

1. You could make the `pipeio_handling()` routine nicer, for instance, buffer a line and only output it when a new line is sent
2. Try wiring up some analog inputs. However, we only have one input on the Core side. How would you multiplex them using available P1 pins?

## Extensions


The more complex example in `t51/example/soc.py` uses some emulation logic to mimic the behaviour of CMOS4xxx logic needed to extend the port I/Os of the actual original hardware behind this mockup. Also, you may notice that the UART interface is in fact a mock up as well and only wraps to a lean pipe interface.

To begin with a proper co-simulation of a UART including proper baud rate, break detection etc., you would typically derive from a `SFRDevice` type class and replace the core creating function (here `.create_uart`) by your own logic. Then, for a proper test, you'd likely reuse the same UART RX/TX logic on the front end side to test proper functionality, robustness against glitches, and what not.

First, we need to elaborate on the SoC architecture:

### SoC


```
             +----------------------+
             |       8051 CPU       |
             |                      |
             |--------------------- |
             |       SFR MAP        |
             +----------+-----------+
                        |
              =====  MMR/SFR BUS  ==============
              |          |          |          |
         +----+---+ +----+---+ +----+---+ +----+---+
         |  GPIO  | |  UART  | |  SPI   | |  TIMER |
         |   IP   | |   IP   | |   IP   | |   IP   |
         +----+---+ +----+---+ +----+---+ +----+---+
              |          |          |          |
           LEDs /     TX / RX    MOSI/MISO    IRQ
           Buttons               SCLK / CS
```

The 8051 core logic implements only a mandatory minimum set of SFRs, such as the `A`, `B` and `PSW` registers, etc.
On top of that, developers insert their own SFR implementation that may deviate from original 8051 implementations.

Basically, this requires editing a register map (see `example/sfr8051_nxp_80c552.py` for instance) and creating the associated peripheral cores. All register settings (bit fields) are wired to a large SFR bus, where IP core wrappers slice their required register portions out of this bus. The actual mapping of the peripheral is again defined in `example/sfr8051.py`. Here, the magic of creating the address decoder happens as well. The important design rule to keep in mind is:

**Make sure bitfields have unique names**

So, if you are implementing two UART ports, you can not use the `TI` flag for both ports. This is a limitation of the flat SFR map with no real hierarchy involved.




### General Architecture




The typical design construct used is represented by a `Factory` class. It
contains testbench, top level unit under test and possibly some target
synthesis rules. Upon construction, it is passed an `Architecture`,
which in most cases is simply a HDL simulator (here: in this notebook we use CXXRTL only).

Let's list a few entry points and hints on how to extend the SFR map or
create a new 8051 derivate.

1) The `T51_Base` class contains the actual implementation of the T51
   cyrite port. It implements the basic requirements of SFRs from the 8051
   specification, plus adds a Monitor structure for later extension.
2) The `.soc_t8052NG` block component extends the core by some logic:
   * External memory and ROM handling
   * Wishbone bus towards outside
   * The SFR interface which is created during instance of the derived
     SoC class
3) The `SFRDevice(Device, SFRMap)` class defines the peripheral map:
   * The SFR map
   * IRQ logic and peripheral wiring
   * Configuration of peripheral cores and logic

The general golden rule behind this structure is:
   * Never change anything in a super class unless it really fixes a bug
   * Add features in derived classes only

This is not yet perfect. You may note platform specific items still residing in some of the base classes. At a later point, this might be cleaned up.

## Licenses, contributions and such

This setup is released with educational goals in mind. Unfortunately, the licensing is unclear on the original code.
However, if you wish to make changes and improvements, please go ahead:

* Fork the project
* Improve, test, optimize
* Once nothing is broken from the notebook perspective, commit, file a PR

Please try to keep AI generated content out of the game as far as you can. It's for education, not for production.